# Notebook 11 — seed / init variance for the silhouette separability tests

The paper's headline separability numbers (the cosine **silhouette** of the BFT
fingerprint vs. dimension-matched activations, and the arbor-NMF vs. activation-NMF
control) are point estimates from a single trace, with error bars / significance
currently coming from a **stimulus bootstrap** of that one trace.

This notebook recomputes those silhouettes across a **more adequate origin of
variance**:

| model | notebook | variance source | replicates |
|---|---|---|---|
| `mlp_even_odd` | nb01 | trained model **seed** | seeds 0–4 |
| `mlp_digit`    | nb02 | trained model **seed** | seeds 0–4 |
| `cnn_cifar`    | nb03 | trained model **seed** | seeds 0–4 |
| `imagenet_cnn` | nb05 | NMF **factorization init** | random_state 0–4 |

SqueezeNet is pretrained, so it has no seed ensemble — its spread comes from the
NMF initialisation instead (`bft(..., random_state=i)`), which is the only
informative spread there.

Every replicate is built with the **exact publication hyperparameters and sample
pipeline of notebooks 01/02/03/05** (not nb09's stale `REG` defaults), so the
per-replicate point estimate matches the paper's figure.

**Output.** One figdata-style bundle per model, `figures/figdata/nb11_<exp>_silhouette.{npz,json}`,
holding the per-replicate silhouette / kNN arrays (both label granularities, all
candidate representations, the shuffled-label null, and the A1 arbor-vs-activation
control). Port these back and recompute error bars (std across replicates) and
significance (paired test across replicates) in place of the bootstrap. A running
JSON is also written to `data/results/nb11_<exp>_silhouette.json`.

Run order is smallest model first; intermediate results are checkpointed after every
replicate, so a killed run is never wasted. Run on the cluster with:

```bash
MPLBACKEND=Agg ./.venv/bin/python scripts/run_nb.py notebooks/11_seed_silhouette.ipynb
# subset / smoke test:
NBSIL_EXPS='mlp_even_odd' NBSIL_MODE=local ./.venv/bin/python scripts/run_nb.py notebooks/11_seed_silhouette.ipynb
```


In [ ]:
import os, sys, json, glob, time, warnings
sys.path.insert(0, '..')

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, TensorDataset

from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA, MiniBatchNMF
from sklearn.random_projection import GaussianRandomProjection
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier

from src import (SimpleMLP, SmallCNN, load_experiment, get_transform,
                 get_loaders_from_config, collect_layer_dicts, bft,
                 extract_fingerprint_matrix, figdata, figexport)
from src.bft import (compute_joint_arbors_normalized, compute_conv_joint_arbors,
                     compute_attn_joint_arbors)
from src.data_utils import label_transformed_loader

warnings.filterwarnings('ignore')

REPO       = os.path.abspath('..')                 # notebook lives in notebooks/
RES_DIR    = os.path.join(REPO, 'data', 'results')
MODEL_ROOT = os.path.join(REPO, 'data', 'models')
os.makedirs(RES_DIR, exist_ok=True)

MODE   = os.environ.get('NBSIL_MODE', 'cluster')   # 'cluster' (publication) | 'local' (smoke)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                      ('mps' if torch.backends.mps.is_available() else 'cpu'))

# 'cluster' reproduces nb01-05 exactly: full sample set, bft's default max_iter (500).
# 'local' caps both for a laptop smoke test — the numbers are NOT publication-grade.
if MODE == 'cluster':
    N_TRACE      = None      # no stimulus cap
    BFT_MAX_ITER = None      # -> bft default (500), matching the notebooks
    AUX_MAX_ITER = 300       # A1 activation/arbor NMF budget (nb09 cluster value)
else:
    N_TRACE      = 400
    BFT_MAX_ITER = 120
    AUX_MAX_ITER = 80

def _bft_iter():
    return {} if BFT_MAX_ITER is None else {'max_iter': BFT_MAX_ITER}

print(f'MODE={MODE}  DEVICE={DEVICE}  N_TRACE={N_TRACE}  BFT_MAX_ITER={BFT_MAX_ITER}')

In [ ]:
# ── Per-model config: EXACT publication hyperparameters + sample pipeline ──────
# Values copied verbatim from notebooks 01/02/03/05 §1 config cells (PUBLICATION
# SETTINGS), NOT nb09's REG (which is stale). If a notebook's config changes,
# update the matching entry here.
REG = {
    'mlp_even_odd': dict(
        kind='mlp', ckpt='mnist_even_odd_mlp_8_4_0134', n_seeds=5,
        arch='SimpleMLP',
        arch_kwargs=dict(input_dim=784, hidden_dims=[8, 4], output_dim=2),
        dataset='MNIST', batch_size=32, digit_filter=[0, 1, 3, 4],
        label='even_odd', n_classes=2, class_names=['even', 'odd'],
        has_fine=True,                       # fine label = original digit
        validate_top_m=2000,
        bft=dict(k_max=[4, 2, 2], n_branches=[1, 1, 2], stimulus_threshold=0.5),
        arch_label='MLP 8-4', ds_label='MNIST even/odd'),

    'mlp_digit': dict(
        kind='mlp', ckpt='mnist_digit_mlp_40_20', n_seeds=5,
        arch='SimpleMLP',
        arch_kwargs=dict(input_dim=784, hidden_dims=[40, 20], output_dim=10),
        dataset='MNIST', batch_size=64, digit_filter=None,
        label='identity', n_classes=10, class_names=[str(i) for i in range(10)],
        has_fine=False,
        validate_top_m=2000,
        bft=dict(k_max=[7, 3, 12], n_branches=[1, 2, 10], stimulus_threshold=0.7),
        arch_label='MLP 40-20', ds_label='MNIST digits'),

    'cnn_cifar': dict(
        kind='cnn', ckpt='cifar10_cnn', n_seeds=5, top_per_class=60,
        arch='SmallCNN',
        arch_kwargs=dict(channels=[32, 64, 128, 256], n_classes=10, global_pool=True),
        n_classes=10, has_fine=False,
        class_names=['airplane', 'automobile', 'bird', 'cat', 'deer',
                     'dog', 'frog', 'horse', 'ship', 'truck'],
        bft=dict(k_max=[4, 3, 4, 6, 10], n_branches=[1, 1, 1, 1, 10],
                 conv_pool_method='avg', stimulus_threshold=0.0),
        arch_label='SmallCNN', ds_label='CIFAR-10'),

    'imagenet_cnn': dict(
        kind='imagenet', n_inits=5, n_per_category=100,
        n_classes=8, has_fine=False,
        class_names=['airplane', 'ship', 'car', 'bicycle',
                     'elephant', 'bear', 'dog', 'bird'],
        bft=dict(k_max=[5, 3, 2, 2, 3, 2, 5, 5, 5, 10],
                 n_branches=[1, 1, 1, 1, 1, 1, 1, 1, 2, 5],
                 conv_pool_method='avg', stimulus_threshold=0.0),
        arch_label='SqueezeNet1.1', ds_label='ImageNet (8 categories)'),
}

# smallest model first; overridable via NBSIL_EXPS='mlp_even_odd cnn_cifar'
EXPS_ORDER = ['mlp_even_odd', 'mlp_digit', 'cnn_cifar', 'imagenet_cnn']
EXPS = os.environ.get('NBSIL_EXPS', '').split() or EXPS_ORDER
print('experiments:', EXPS)

In [ ]:
# ── Silhouette / separability helpers — identical convention to nb09 §S4/§S5 ───
# sklearn silhouette on L2-normalized rows (monotone with cosine), plus a small
# cross-validated kNN accuracy. This is the convention that produced the paper's
# reported separability numbers.
def knn_cv(X, y, kmax=5, cv_max=3):
    classes, counts = np.unique(y, return_counts=True)
    mn = int(counts.min())
    if len(classes) < 2 or mn < 2:
        return float('nan')
    cv = int(min(cv_max, mn))
    k  = int(max(1, min(kmax, mn)))
    return float(cross_val_score(KNeighborsClassifier(k), X, y, cv=cv).mean())


def sep_metrics(X, y):
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
    if len(np.unique(y)) < 2 or len(y) < 12:
        return float('nan'), float('nan')
    return float(silhouette_score(Xn, y)), knn_cv(Xn, y)


def fit_nmf(X, k, rs):
    """Activation/arbor NMF for the A1 control. rs = the replicate's variance seed."""
    Xc = np.clip(X, 0, None).astype(np.float32)
    n  = Xc.shape[0]
    sub = Xc if n <= 800 else Xc[np.random.default_rng(0).choice(n, 800, replace=False)]
    m = MiniBatchNMF(n_components=k, random_state=rs, max_iter=AUX_MAX_ITER,
                     batch_size=1024, init='random')
    m.fit(sub)
    return m.transform(Xc)


def act_matrix_for(nd, layer_inputs):
    """Activation-only matrix (N, features): pool conv spatial / collapse attn tokens."""
    li = layer_inputs[nd.layer_idx]
    if nd.layer_type == 'attn' and getattr(nd, 'attn_weights', None) is not None:
        return np.einsum('nt,ntd->nd', nd.attn_weights, li)
    if li.ndim == 4:
        return li.mean(axis=(2, 3))
    return li.reshape(len(li), -1)


def node_arbor_pos(nd, layer_inputs, pool_method):
    """Positive half of a node's signed joint arbor (what BFT's NMF is fit to)."""
    li = layer_inputs[nd.layer_idx]
    if nd.layer_type == 'conv':
        X = compute_conv_joint_arbors(nd.weight, li, stimulus_weights=nd.stimulus_weights,
                                      stimulus_threshold=nd.stimulus_threshold,
                                      pool_method=pool_method)
    elif nd.layer_type == 'attn':
        X = compute_attn_joint_arbors(nd.weight, li, nd.attn_weights,
                                      stimulus_weights=nd.stimulus_weights,
                                      stimulus_threshold=nd.stimulus_threshold)
    else:
        X = compute_joint_arbors_normalized(nd.weight, li,
                                            stimulus_weights=nd.stimulus_weights,
                                            stimulus_threshold=nd.stimulus_threshold)
    return np.clip(X, 0, None)


CANDS = ['bft_fingerprint', 'raw_activations', 'bft_matched', 'act_matched', 'act_randproj']


def separability_panel(tree, layer_inputs, targets, fine, has_fine):
    """Reproduce nb09 §S4: silhouette/kNN of the BFT fingerprint vs a dimension-
    matched pooled-activation baseline, by task and (if any) fine label, plus a
    shuffled-label null. Returns per-candidate scalars for one replicate."""
    n = len(targets)
    F = extract_fingerprint_matrix(tree, np.arange(n))
    nodes_by_layer = {}
    for nd in tree.nodes():
        nodes_by_layer.setdefault(nd.layer_idx, nd)
    layer_ids = sorted(nodes_by_layer)
    _alayers = layer_ids[1:] if len(layer_ids) > 1 else layer_ids
    A = np.concatenate([act_matrix_for(nodes_by_layer[i], layer_inputs) for i in _alayers], axis=1)

    d_match = int(min(F.shape[1], A.shape[1], max(2, n - 1)))
    _pca = lambda M: M if M.shape[1] == d_match else PCA(d_match, random_state=0).fit_transform(M)
    cands = {'bft_fingerprint': F, 'raw_activations': A,
             'bft_matched': _pca(F), 'act_matched': _pca(A)}
    if A.shape[1] > d_match:
        cands['act_randproj'] = GaussianRandomProjection(d_match, random_state=0).fit_transform(A)

    out = {'by_task': {}, 'by_fine': {},
           'dims': {'fingerprint': int(F.shape[1]), 'activations': int(A.shape[1]),
                    'matched': d_match}}
    for name, X in cands.items():
        s_, k_ = sep_metrics(X, targets)
        out['by_task'][name] = {'silhouette': s_, 'knn_acc': k_}
        if has_fine:
            s2, k2 = sep_metrics(X, fine)
            out['by_fine'][name] = {'silhouette': s2, 'knn_acc': k2}
    ysh = targets.copy()
    np.random.default_rng(0).shuffle(ysh)
    s0, k0 = sep_metrics(F, ysh)
    out['null_shuffled_labels'] = {'silhouette': s0, 'knn_acc': k0}
    return out, nodes_by_layer, layer_ids


def a1_panel(tree, layer_inputs, targets, nodes_by_layer, layer_ids, pool_method, rs):
    """Reproduce nb09 §S5 fingerprint_separability: silhouette of the W·a arbor-NMF
    fingerprint vs the activation-only NMF fingerprint (the 'does multiplying in the
    weights buy separability' control). NMF init = rs so imagenet inits perturb it too."""
    afp, cfp = [], []
    for li_ in layer_ids:
        nd = nodes_by_layer[li_]
        k  = nd.img_factors.shape[1]
        Xa = node_arbor_pos(nd, layer_inputs, pool_method)
        Ac = np.clip(act_matrix_for(nd, layer_inputs), 0, None)
        kA = max(1, min(k, Ac.shape[1]))
        afp.append(fit_nmf(Xa, k, rs))
        cfp.append(fit_nmf(Ac, kA, rs))
    sA, kA_ = sep_metrics(np.concatenate(afp, 1), targets)
    sC, kC_ = sep_metrics(np.concatenate(cfp, 1), targets)
    return {'arbor_nmf': {'silhouette': sA, 'knn_acc': kA_},
            'activation_nmf': {'silhouette': sC, 'knn_acc': kC_}}


def score_replicate(tree, layer_inputs, targets, fine, has_fine, pool_method, rs):
    """Full per-replicate score: separability panel + A1 control + causal R² if the
    tree carries a validation summary (primary-mode MLPs only)."""
    targets = np.asarray(targets).astype(int)
    fine    = np.asarray(fine).astype(int)
    sep, nbl, lids = separability_panel(tree, layer_inputs, targets, fine, has_fine)
    a1 = a1_panel(tree, layer_inputs, targets, nbl, lids, pool_method, rs)
    vs = None
    try:
        vs = tree.validation_summary()
    except Exception:
        vs = None
    pre = vs['overall']['preact_r2'] if vs else None
    return {'n_samples': int(len(targets)),
            'n_classes': int(len(np.unique(targets))),
            'sep': sep, 'a1': a1,
            'min_preact_r2': float(pre['min']) if pre else float('nan'),
            'median_preact_r2': float(pre['median']) if pre else float('nan')}

In [ ]:
# ── Per-model replicate builders — each returns (tree, layer_inputs, targets, fine) ─
# using the SAME loader / sample pipeline as the source notebook. layer_inputs is
# row-aligned with the tree's fingerprint rows (only_correct=True on both sides).

def _cap(loader, batch=256):
    if N_TRACE is None:
        return loader
    ds = loader.dataset
    return DataLoader(Subset(ds, list(range(min(N_TRACE, len(ds))))),
                      batch_size=batch, shuffle=False)


def build_mlp(r, model, test_loader, label_transform, rs):
    tl = _cap(test_loader)
    coll = collect_layer_dicts(model, tl, label_transform=label_transform, device=DEVICE)
    layer_inputs = [d['input_fmap'] for d in coll['layer_data']]
    targets = coll['targets']
    fine    = coll.get('digits', coll['targets'])
    vloader = label_transformed_loader(tl, label_transform) if label_transform else tl
    tree = bft(model, vloader, **r['bft'], weighting='img_selectivity',
               validate=True, validate_top_m=r['validate_top_m'],
               n_jobs=3, random_state=rs, **_bft_iter())
    return tree, layer_inputs, targets, fine


def _cnn_confidence_filter(raw, top_k, n_classes):
    keep = np.sort(np.concatenate([
        np.where(raw['targets'] == c)[0][
            np.argsort(raw['confidences'][raw['targets'] == c])[::-1][:top_k]]
        for c in range(n_classes)]))
    layer_data = [{**ld, 'input_fmap': ld['input_fmap'][keep],
                   'output_fmap': ld['output_fmap'][keep]} for ld in raw['layer_data']]
    return {'images': raw['images'][keep], 'targets': raw['targets'][keep],
            'layer_data': layer_data}


def build_cnn(r, model, test_loader, rs):
    if N_TRACE is not None:
        test_loader = _cap(test_loader)
    raw = collect_layer_dicts(model, test_loader, DEVICE, only_correct=True)
    data0 = _cnn_confidence_filter(raw, r['top_per_class'], r['n_classes'])
    layer_inputs = [ld['input_fmap'] for ld in data0['layer_data']]
    targets = data0['targets']
    tree = bft(data0['layer_data'], **r['bft'], weighting='img_selectivity',
               n_jobs=3, random_state=rs, **_bft_iter())
    return tree, layer_inputs, targets, targets


# ── ImageNet: collect the spine layer-dicts ONCE, refactorize per init ────────
def imagenet_collect(r):
    import torchvision.transforms as T
    from torchvision import datasets
    from torchvision.models import squeezenet1_1, SqueezeNet1_1_Weights
    CATS = {'airplane': [404, 895], 'ship': [403, 724], 'car': [609, 751],
            'bicycle': [444, 671], 'elephant': [101, 385], 'bear': [294, 297],
            'dog': [151, 251], 'bird': [7, 9]}
    idx2cat = {ii: ci for ci, c in enumerate(CATS) for ii in CATS[c]}
    model = squeezenet1_1(weights=SqueezeNet1_1_Weights.IMAGENET1K_V1).to(DEVICE).eval()
    tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(),
                    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
    try:
        ds = datasets.ImageNet('../data', split='val', transform=tf)
        tgts = np.array(ds.targets)
    except Exception:
        root = '../data/val'
        if not os.path.isdir(root):
            raise RuntimeError('imagenet_cnn needs ImageNet val data at ../data/val '
                               '(ImageFolder) or a torchvision ImageNet root at ../data.')
        ds = datasets.ImageFolder(root, transform=tf)
        tgts = np.array([t for _, t in ds.samples])
    focus = np.where(np.isin(tgts, list(idx2cat)))[0]
    floader = DataLoader(Subset(ds, focus), batch_size=64, shuffle=False, num_workers=4)

    def spine(name, mod):
        return name in ('features.0', 'classifier.1') or (
            isinstance(mod, nn.Conv2d) and name.endswith('.squeeze'))

    raw = collect_layer_dicts(model, floader, device=DEVICE, only_correct=True,
                              layer_filter=spine)
    cat_tgts = np.array([idx2cat.get(int(t), -1) for t in raw['targets']])
    # top-N most confident per category (nb05 filter_by_category)
    keep = []
    conf = raw.get('confidences')
    for ci in range(r['n_classes']):
        ci_idx = np.where(cat_tgts == ci)[0]
        if conf is not None and len(ci_idx):
            ci_idx = ci_idx[np.argsort(conf[ci_idx])[::-1]]
        keep.append(ci_idx[:r['n_per_category']])
    keep = np.sort(np.concatenate(keep))
    if N_TRACE is not None and len(keep) > N_TRACE:
        keep = np.sort(np.random.default_rng(0).choice(keep, N_TRACE, replace=False))
    layer_data = [{**ld, 'input_fmap': ld['input_fmap'][keep]} for ld in raw['layer_data']]
    targets = cat_tgts[keep]
    layer_inputs = [ld['input_fmap'] for ld in layer_data]
    return layer_data, layer_inputs, targets


def build_imagenet_init(r, layer_data, rs):
    tree = bft(layer_data, **r['bft'], weighting='img_selectivity',
               n_jobs=3, random_state=rs, **_bft_iter())
    return tree

In [ ]:
# ── Aggregation + bundle assembly ─────────────────────────────────────────────
def _series(rows, group, cand, metric):
    return [float(row['sep'].get(group, {}).get(cand, {}).get(metric, np.nan)) for row in rows]


def build_bundle(exp, r, rep_type, ids, rows):
    n_rep = len(rows)
    sep = {'by_task': {}, 'by_fine': {}}
    for cand in CANDS:
        sep['by_task'][cand] = {'silhouette': _series(rows, 'by_task', cand, 'silhouette'),
                                'knn_acc':    _series(rows, 'by_task', cand, 'knn_acc')}
        if r['has_fine']:
            sep['by_fine'][cand] = {'silhouette': _series(rows, 'by_fine', cand, 'silhouette'),
                                    'knn_acc':    _series(rows, 'by_fine', cand, 'knn_acc')}
    sep['null_shuffled_labels'] = {
        'silhouette': [float(row['sep']['null_shuffled_labels']['silhouette']) for row in rows],
        'knn_acc':    [float(row['sep']['null_shuffled_labels']['knn_acc']) for row in rows]}
    sep['dims'] = {kk: [int(row['sep']['dims'][kk]) for row in rows]
                   for kk in ('fingerprint', 'activations', 'matched')}
    a1 = {'arbor_nmf':      {'silhouette': [float(row['a1']['arbor_nmf']['silhouette']) for row in rows],
                             'knn_acc':    [float(row['a1']['arbor_nmf']['knn_acc']) for row in rows]},
          'activation_nmf': {'silhouette': [float(row['a1']['activation_nmf']['silhouette']) for row in rows],
                             'knn_acc':    [float(row['a1']['activation_nmf']['knn_acc']) for row in rows]}}

    def ms(v):
        v = np.asarray(v, float); v = v[~np.isnan(v)]
        return (float(v.mean()), float(v.std())) if v.size else (float('nan'), float('nan'))

    fp_m, fp_s   = ms(sep['by_task']['bft_fingerprint']['silhouette'])
    act_m, act_s = ms(sep['by_task']['act_matched']['silhouette'])
    ar_m, ar_s   = ms(a1['arbor_nmf']['silhouette'])
    an_m, an_s   = ms(a1['activation_nmf']['silhouette'])
    summary = dict(
        fp_silhouette_mean=fp_m, fp_silhouette_std=fp_s,
        act_matched_silhouette_mean=act_m, act_matched_silhouette_std=act_s,
        arbor_nmf_silhouette_mean=ar_m, arbor_nmf_silhouette_std=ar_s,
        activation_nmf_silhouette_mean=an_m, activation_nmf_silhouette_std=an_s,
        fp_minus_act_matched_mean=fp_m - act_m,
        arbor_minus_activation_nmf_mean=ar_m - an_m)

    return dict(
        exp=exp, label=r['ds_label'], arch=r['arch_label'],
        replicate_type=rep_type, replicate_ids=list(ids), n_replicates=n_rep,
        n_classes=r['n_classes'], class_names=list(r['class_names']),
        has_fine=bool(r['has_fine']),
        n_samples=[int(row['n_samples']) for row in rows],
        candidates=CANDS,
        bft_k_max=list(r['bft']['k_max']),
        bft_n_branches=list(r['bft']['n_branches']),
        bft_stimulus_threshold=float(r['bft']['stimulus_threshold']),
        bft_conv_pool_method=r['bft'].get('conv_pool_method', 'none'),
        sep=sep, a1=a1,
        min_preact_r2=[float(row['min_preact_r2']) for row in rows],
        median_preact_r2=[float(row['median_preact_r2']) for row in rows],
        summary=summary,
        source=('notebook 11 — publication HPs from source notebook; '
                f'variance across {rep_type} (n={n_rep})'),
        mode=MODE)


def checkpoint_json(exp, rep_type, ids_done, rows, done=False):
    path = os.path.join(RES_DIR, f'nb11_{exp}_silhouette.json')
    payload = dict(exp=exp, replicate_type=rep_type, mode=MODE,
                   replicates_done=list(ids_done), complete=bool(done), rows=rows)
    tmp = path + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(payload, f, indent=1, default=float)
    os.replace(tmp, path)
    print(f'  [checkpoint] {exp}: {len(ids_done)} replicate(s) -> '
          f'{os.path.relpath(path, REPO)}{"  (DONE)" if done else ""}')

In [ ]:
# ── Run: all requested experiments, smallest first, checkpoint per replicate ───
SUMMARY = {}
for exp in EXPS:
    r = REG[exp]
    print('\n' + '=' * 78 + f'\n{exp}  [{r["arch_label"]} / {r["ds_label"]}]  kind={r["kind"]}')
    t0 = time.time()
    rows, ids_done = [], []
    try:
        if r['kind'] == 'mlp':
            rep_type = 'model_seed'
            cfg = {'arch': r['arch'], 'arch_kwargs': r['arch_kwargs'], 'dataset': r['dataset'],
                   'dataset_kwargs': {'root': '../data/', 'batch_size': r['batch_size'],
                                      **({'digit_filter': r['digit_filter']} if r['digit_filter'] else {})},
                   'label_transform': r['label']}
            _, test_loader = get_loaders_from_config(cfg)
            label_transform = get_transform(r['label'])
            seeds = [s for s in range(r['n_seeds'])
                     if os.path.exists(os.path.join(MODEL_ROOT, f"{r['ckpt']}_seed{s}", 'weights.pt'))]
            if not seeds:
                raise RuntimeError(f'no {r["ckpt"]}_seed* checkpoints under {MODEL_ROOT}')
            print(f'  seeds on disk: {seeds}')
            for s in seeds:
                model = load_experiment(os.path.join(MODEL_ROOT, f"{r['ckpt']}_seed{s}"), DEVICE)[0]
                tree, li, tg, fn = build_mlp(r, model, test_loader, label_transform, rs=0)
                row = score_replicate(tree, li, tg, fn, r['has_fine'],
                                      r['bft'].get('conv_pool_method', 'avg'), rs=0)
                row['seed'] = s
                rows.append(row); ids_done.append(s)
                print(f'  seed {s}: fp_sil(task)={row["sep"]["by_task"]["bft_fingerprint"]["silhouette"]:.3f}  '
                      f'act_matched={row["sep"]["by_task"]["act_matched"]["silhouette"]:.3f}  '
                      f'arbor_nmf={row["a1"]["arbor_nmf"]["silhouette"]:.3f}')
                checkpoint_json(exp, rep_type, ids_done, rows)

        elif r['kind'] == 'cnn':
            rep_type = 'model_seed'
            import torchvision, torchvision.transforms as T
            tf = T.Compose([T.ToTensor(), T.Normalize((0.4914, 0.4822, 0.4465),
                                                      (0.2470, 0.2435, 0.2616))])
            test_ds = torchvision.datasets.CIFAR10('../data', train=False, download=True, transform=tf)
            test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)
            seeds = [s for s in range(r['n_seeds'])
                     if os.path.exists(os.path.join(MODEL_ROOT, f"{r['ckpt']}_seed{s}", 'weights.pt'))]
            if not seeds:
                raise RuntimeError(f'no {r["ckpt"]}_seed* checkpoints under {MODEL_ROOT}')
            print(f'  seeds on disk: {seeds}')
            for s in seeds:
                model = load_experiment(os.path.join(MODEL_ROOT, f"{r['ckpt']}_seed{s}"), DEVICE)[0]
                tree, li, tg, fn = build_cnn(r, model, test_loader, rs=0)
                row = score_replicate(tree, li, tg, fn, r['has_fine'],
                                      r['bft'].get('conv_pool_method', 'avg'), rs=0)
                row['seed'] = s
                rows.append(row); ids_done.append(s)
                print(f'  seed {s}: fp_sil(task)={row["sep"]["by_task"]["bft_fingerprint"]["silhouette"]:.3f}  '
                      f'act_matched={row["sep"]["by_task"]["act_matched"]["silhouette"]:.3f}  '
                      f'arbor_nmf={row["a1"]["arbor_nmf"]["silhouette"]:.3f}')
                checkpoint_json(exp, rep_type, ids_done, rows)

        elif r['kind'] == 'imagenet':
            rep_type = 'nmf_init'
            print('  collecting SqueezeNet spine layer-dicts (once) …')
            layer_data, li, tg = imagenet_collect(r)
            print(f'  {len(tg)} samples | {len(li)} spine layers')
            for init in range(r['n_inits']):
                tree = build_imagenet_init(r, layer_data, rs=init)
                row = score_replicate(tree, li, tg, tg, r['has_fine'],
                                      r['bft'].get('conv_pool_method', 'avg'), rs=init)
                row['init'] = init
                rows.append(row); ids_done.append(init)
                print(f'  init {init}: fp_sil(task)={row["sep"]["by_task"]["bft_fingerprint"]["silhouette"]:.3f}  '
                      f'act_matched={row["sep"]["by_task"]["act_matched"]["silhouette"]:.3f}  '
                      f'arbor_nmf={row["a1"]["arbor_nmf"]["silhouette"]:.3f}')
                checkpoint_json(exp, rep_type, ids_done, rows)
        else:
            raise NotImplementedError(r['kind'])

        checkpoint_json(exp, rep_type, ids_done, rows, done=True)
        bundle = build_bundle(exp, r, rep_type, ids_done, rows)
        figdata.save(f'nb11_{exp}_silhouette', bundle)
        SUMMARY[exp] = bundle['summary']
        print(f'  bundle nb11_{exp}_silhouette written | {time.time() - t0:.0f}s')

    except Exception as e:
        import traceback
        print(f'  [SKIP] {exp}: {type(e).__name__}: {e}')
        traceback.print_exc()
        SUMMARY[exp] = {'error': f'{type(e).__name__}: {e}'}

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
print('\n' + '=' * 78 + '\nSilhouette (mean ± std across replicates)\n' + '=' * 78)
for exp in EXPS:
    s = SUMMARY.get(exp, {})
    if 'error' in s:
        print(f'{exp:16s}  ERROR: {s["error"]}')
        continue
    if not s:
        print(f'{exp:16s}  (not run)')
        continue
    print(f'{exp:16s}  BFT fp {s["fp_silhouette_mean"]:.3f} ± {s["fp_silhouette_std"]:.3f}  |  '
          f'act-matched {s["act_matched_silhouette_mean"]:.3f} ± {s["act_matched_silhouette_std"]:.3f}  '
          f'(Δ={s["fp_minus_act_matched_mean"]:+.3f})  ||  '
          f'arbor-NMF {s["arbor_nmf_silhouette_mean"]:.3f} vs act-NMF '
          f'{s["activation_nmf_silhouette_mean"]:.3f} (Δ={s["arbor_minus_activation_nmf_mean"]:+.3f})')
print('\nBundles:  figures/figdata/nb11_<exp>_silhouette.{npz,json}')
print('JSON:     data/results/nb11_<exp>_silhouette.json')